# V2 Pose Optimization (Next Best View)
This notebook uses the V2 Global Chamfer Distances to run a Multi-Objective Optimization Problem (MOOP) using a GRASP algorithm.

In [13]:
import os
import json
import pandas as pd
import numpy as np
import random
import time

# ==========================================
# 1. CONFIGURATION
# ==========================================
EXPERIMENT = "test_8_simulation2"
WORKPIECE = "TH0011AV"

# MOOP Weights
ALPHA = 0.3  # Weight for Coverability (Information Gain)
BETA = 0.7   # Weight for Confidence (Quality)

TOP_PERCENTILE_FILTER = 1.0 # Keep top n*100% of viewpoints based on Score
NUM_ITERATIONS = 10
NUM_VIEWS_TO_SELECT = 4


In [14]:
# ==========================================
# 2. DATA INGESTION & FILTERING
# ==========================================
csv_path = f"processed_data/{EXPERIMENT}/V2_chamfer_results.csv"
df_chamfer = pd.read_csv(csv_path)
df_chamfer = df_chamfer[df_chamfer['Workpiece'] == WORKPIECE].copy()

cov_json = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{WORKPIECE}/covered_indices.json"
with open(cov_json, "r") as f:
    global_visibility_dict = json.load(f)

# Determine total possible points across the object
all_possible_indices = set()
for v_idx in df_chamfer['Viewpoint']:
    visible = global_visibility_dict.get(str(v_idx), [])
    all_possible_indices.update(visible)
total_object_points = len(all_possible_indices)

# Normalize Confidence for this workpiece
min_cd = df_chamfer['Chamfer_Distance_mm'].min()
max_cd = df_chamfer['Chamfer_Distance_mm'].max()

if max_cd > min_cd:
    df_chamfer['Confidence'] = 1.0 - ((df_chamfer['Chamfer_Distance_mm'] - min_cd) / (max_cd - min_cd))
else:
    df_chamfer['Confidence'] = 1.0

# Calculate custom score for the Top-20% Filter
scores = []
for _, row in df_chamfer.iterrows():
    v_idx = str(int(row['Viewpoint']))
    points_seen = len(global_visibility_dict.get(v_idx, []))
    score = (points_seen / total_object_points) * row['Confidence']
    scores.append(score)
    
df_chamfer['Filter_Score'] = scores

# Apply Top-20% Filter
num_to_keep = max(1, int(len(df_chamfer) * TOP_PERCENTILE_FILTER))
df_filtered = df_chamfer.nlargest(num_to_keep, 'Filter_Score').copy()

print(f"Data Ingestion Complete for {WORKPIECE}!")
print(f"Original Cameras: {len(df_chamfer)} | Filtered Cameras: {len(df_filtered)}")


Data Ingestion Complete for TH0011AV!
Original Cameras: 432 | Filtered Cameras: 432


In [15]:
# ==========================================
# 3. GRASP OPTIMIZATION LOOP
# ==========================================
best_global_utility = -float('inf')
best_sequence = []
best_sequence_breakdown = []

print("Running GRASP Optimizer...")
start_time = time.time()

for iteration in range(1, NUM_ITERATIONS + 1):
    covered_indices = set()
    current_sequence = []
    current_breakdown = []
    total_utility = 0.0
    
    # Clone the filtered pool for this iteration
    pool = df_filtered.copy()
    
    for step in range(NUM_VIEWS_TO_SELECT):
        if pool.empty: break
            
        # 1. Calculate Marginal Coverability
        marginal_gains = []
        for _, row in pool.iterrows():
            v_idx = str(int(row['Viewpoint']))
            visible = set(global_visibility_dict.get(v_idx, []))
            new_points = len(visible - covered_indices)
            marginal_gains.append(new_points)
        
        pool['Marginal_Coverability'] = marginal_gains
        
        # 2. Normalize Marginal Coverability [0, 1]
        min_cov = pool['Marginal_Coverability'].min()
        max_cov = pool['Marginal_Coverability'].max()
        if max_cov > min_cov:
            pool['Norm_Coverability'] = (pool['Marginal_Coverability'] - min_cov) / (max_cov - min_cov)
        else:
            pool['Norm_Coverability'] = 1.0
            
        # 3. Calculate MOOP Utility
        pool['Utility_Score'] = (ALPHA * pool['Norm_Coverability']) + (BETA * pool['Confidence'])
        
        # 4. GRASP Selection (Pick randomly from Top 3)
        pool_sorted = pool.sort_values(by='Utility_Score', ascending=False)
        top_k = pool_sorted.head(3)
        selected_row = top_k.sample(n=1).iloc[0]
        
        # 5. Update State
        selected_v_idx = int(selected_row['Viewpoint'])
        current_sequence.append(selected_v_idx)
        total_utility += selected_row['Utility_Score']
        
        visible_pts = set(global_visibility_dict.get(str(selected_v_idx), []))
        covered_indices.update(visible_pts)
        
        current_breakdown.append({
            'Step': step + 1,
            'Viewpoint': selected_v_idx,
            'Utility': round(selected_row['Utility_Score'], 4),
            'Cumulative Coverability': round(len(covered_indices) / total_object_points, 4),
            'Raw Error (mm)': round(selected_row['Chamfer_Distance_mm'], 4),
            'Normalized Confidence': round(selected_row['Confidence'], 4)
        })
        
        # Remove selected from pool
        pool = pool[pool['Viewpoint'] != selected_v_idx]
        
    if total_utility > best_global_utility:
        best_global_utility = total_utility
        best_sequence = current_sequence
        best_sequence_breakdown = current_breakdown

print(f"Optimization complete in {time.time() - start_time:.1f}s")
print(f"\nBest Sequence Found: {best_sequence}")
print(f"Total Utility: {best_global_utility:.4f}")

from IPython.display import display
print("\nBreakdown of Best Sequence:")
display(pd.DataFrame(best_sequence_breakdown).set_index('Step'))


Running GRASP Optimizer...
Optimization complete in 17.8s

Best Sequence Found: [19, 152, 157, 137]
Total Utility: 3.3015

Breakdown of Best Sequence:


,Viewpoint,Utility,Cumulative Coverability,Raw Error (mm),Normalized Confidence
Step,,,,,
1,19,0.8703,0.6394,2.0012,0.8170
2,152,0.8431,0.8457,1.9678,0.9272
3,157,0.8319,0.9548,2.0065,0.7995
4,137,0.7563,0.9661,2.0489,0.6597


In [16]:
# ==========================================
# 4. VISUALIZATION PREPARATION
# ==========================================
import open3d as o3d
import re
import copy
import matplotlib.pyplot as plt

ENABLE_VISUALIZATION = True
GLOBAL_PCD_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{WORKPIECE}/pcd_all.pcd"
POSES_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{WORKPIECE}"

pcd_all = o3d.io.read_point_cloud(GLOBAL_PCD_PATH)
pcd_all.paint_uniform_color([0.6, 0.6, 0.6])

def make_arrow(direction='x', size=50.0, color=(0.6, 0.6, 0.6)):
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=size * 0.05,
        cone_radius=size * 0.15,
        cylinder_height=size * 0.80,
        cone_height=size * 0.20,
        resolution=20,
    )
    if direction == 'x':
        R_align = arrow.get_rotation_matrix_from_xyz((0, np.pi / 2, 0))
        arrow.rotate(R_align, center=(0, 0, 0))
    arrow.paint_uniform_color(list(color))
    arrow.compute_vertex_normals()
    return arrow

def make_xz_arrows(transform=None, size=50.0, color=(0.6, 0.6, 0.6)):
    frame = make_arrow('x', size=size, color=color) + make_arrow('z', size=size, color=color)
    if transform is not None:
        frame.transform(transform)
    return frame

def load_viewpoint_poses_dict(folder_path):
    def extract_number(filename):
        match = re.search(r'viewpoint_pose_(\d+)\.npy', filename)
        return int(match.group(1)) if match else -1
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    poses = {}
    for f in npy_files:
        idx = extract_number(f)
        if idx != -1:
            poses[idx] = np.load(os.path.join(folder_path, f))
    return poses

poses_dict = load_viewpoint_poses_dict(POSES_PATH)

# Re-evaluate sequence to get step-by-step covered indices
step_to_covered_indices = {}
covered_so_far = set()

# Change this to MANUAL_SEQUENCE if you want to visualize your manual grids!
sequence_to_visualize = best_sequence 

for k, v_idx in enumerate(sequence_to_visualize, 1):
    visible = set(global_visibility_dict.get(str(v_idx), []))
    new_points = visible - covered_so_far
    step_to_covered_indices[k] = new_points
    covered_so_far.update(visible)


In [17]:
# ==========================================
# 5. VISUALIZATION COVERAGE PER VIEWPOINT
# ==========================================
vis_pcd = copy.deepcopy(pcd_all)
colors = np.ones((len(vis_pcd.points), 3)) * 0.8
cmap = plt.get_cmap("tab10")

print("Point Colors:")
for k in range(1, len(sequence_to_visualize) + 1):
    if k in step_to_covered_indices:
        step_color = cmap(k - 1)[:3]
        r, g, b = [int(c * 255) for c in step_color]
        colored_text = f"\033[38;2;{r};{g};{b}mStep {k}\033[0m"
        print(f"{colored_text} covered {len(step_to_covered_indices[k])} new points.")
        indices = list(step_to_covered_indices[k])
        colors[indices] = step_color

vis_pcd.colors = o3d.utility.Vector3dVector(colors)

print("\nOpening Open3D visualization window...")
if ENABLE_VISUALIZATION:
    o3d.visualization.draw_geometries(
        [vis_pcd], 
        window_name="Step-by-Step Coverage",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Point Colors:
Step 1 covered 21190 new points.
Step 2 covered 6837 new points.
Step 3 covered 3617 new points.
Step 4 covered 374 new points.

Opening Open3D visualization window...


In [20]:
# ==========================================
# 6. VISUALIZE SELECTED GRASP POSES
# ==========================================
vis_pcd_poses = copy.deepcopy(pcd_all)
print("Generating visualization of the entire workpiece and camera poses...")
geometries = [vis_pcd_poses]

print("\n--- Selected Viewpoints Summary ---")
for k, v_idx in enumerate(sequence_to_visualize, 1):
    step_color = cmap(k - 1)[:3]
    r, g, b = [int(c * 255) for c in step_color]
    colored_text = f"\033[38;2;{r};{g};{b}mStep {k}\033[0m"
    print(f"{colored_text}: Viewpoint {v_idx}")
    
    if v_idx in poses_dict:
        pose_arrows = make_xz_arrows(transform=poses_dict[v_idx], size=20.0, color=step_color)
        geometries.append(pose_arrows)

print("\nOpening Open3D visualization window...")
if ENABLE_VISUALIZATION:
    o3d.visualization.draw_geometries(
        geometries, 
        window_name="Entire Workpiece and Camera Poses",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Generating visualization of the entire workpiece and camera poses...

--- Selected Viewpoints Summary ---
Step 1: Viewpoint 19
Step 2: Viewpoint 152
Step 3: Viewpoint 157
Step 4: Viewpoint 137

Opening Open3D visualization window...


In [19]:
# ==========================================
# 10. EVALUATE MANUAL SEQUENCE
# ==========================================
MANUAL_SEQUENCE = [80, 84, 88, 92]

covered_indices = set()
manual_breakdown = []
total_util = 0.0

for step, v_idx in enumerate(MANUAL_SEQUENCE):
    row = df_chamfer[df_chamfer['Viewpoint'] == v_idx]
    if row.empty:
        print(f"Warning: Viewpoint {v_idx} not found in dataset!")
        continue
    row = row.iloc[0]
    
    visible_pts = set(global_visibility_dict.get(str(v_idx), []))
    new_points = len(visible_pts - covered_indices)
    covered_indices.update(visible_pts)
    
    cum_cov = len(covered_indices) / total_object_points
    # Note: Utility here is Cumulative, not Marginal, so it's not comparable to Block 3!
    util = (ALPHA * cum_cov) + (BETA * row['Confidence'])
    total_util += util
    
    manual_breakdown.append({
        'Step': step + 1,
        'Viewpoint': v_idx,
        'Utility (Cumulative)': round(util, 4),
        'Cumulative Coverability': round(cum_cov, 4),
        'Raw Error (mm)': round(row['Chamfer_Distance_mm'], 4),
        'Normalized Confidence': round(row['Confidence'], 4)
    })

print(f"Manual Sequence Evaluation: {MANUAL_SEQUENCE}")
print(f"Total Utility (Cumulative): {total_util:.4f}")
display(pd.DataFrame(manual_breakdown).set_index('Step'))


Manual Sequence Evaluation: [80, 84, 88, 92]
Total Utility (Cumulative): 2.6782


,Viewpoint,Utility (Cumulative),Cumulative Coverability,Raw Error (mm),Normalized Confidence
Step,,,,,
1,80,0.5982,0.5099,2.0560,0.6360
2,84,0.6339,0.6942,2.0645,0.6081
3,88,0.7004,0.8623,2.0576,0.6310
4,92,0.7457,0.9725,2.0522,0.6486
